# RFM Analysis

In [1]:
#set colors 
import matplotlib.pyplot as plt

# ── Theme colors ───────────────────────────────────────────
GES_DARK  = '#01010b'   # black
GES_BLUE  = '#6950ec'   # purple
GES_GOLD  = '#e00e32'   # berry red
GES_TEAL  = '#a3055c'   # maroon

PALETTE = [
    '#6950ec',  # purple
    '#a3055c',  # maroon
    '#e00e32',  # berry red
    '#9e9e9e',  # gray
    '#8a7cff',  # soft purple
    '#c75a93',  # dusty pink
    '#d9d9d9',  # light gray
    '#b23a6f'   # muted berry
]

# ── Global chart settings ──────────────────────────────────
FIGSIZE = (10, 6)
TITLE_SIZE = 14
LABEL_SIZE = 11
TICK_SIZE = 10
BAR_LABEL_SIZE = 9
ROTATION = 20
GRID_ALPHA = 0.35
BAR_ALPHA = 0.9
EDGE_COLOR = '#d9d9d9'

plt.rcParams.update({
    'figure.facecolor':  GES_DARK,
    'axes.facecolor':    '#141420',
    'savefig.facecolor': GES_DARK,
    'axes.edgecolor':    '#444',
    'axes.labelcolor':   'white',
    'xtick.color':       '#ccc',
    'ytick.color':       '#ccc',
    'text.color':        'white',
    'grid.color':        '#333',
    'grid.linestyle':    '--',
    'grid.alpha':        0.5,
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

## Import Data and Creating RFM metric from Jonny's notebook

In [10]:
import pandas as pd

df = pd.read_csv("Data/merged_customer_data.csv")
df.head()

,Transaction ID,Customer ID,Transaction Date,Business Line,Product/Service,Revenue,Data Source,Region,Source Type,Primary Business Line,...,Source ID,Business Line_ds,Data Source Name,Source Type_ds,Customer ID Available,Email Available,Demographics Available,Data Quality,Integration Status,Notes
0,TXN000001,CUST00150,2024-01-01,Consumer Products,Merchandise - Collectibles,54.06,3rd Party - Walmart,East Coast,3rd Party,Consumer Products,...,8,Consumer Products,3rd Party - Walmart,3rd Party,No,No,No,Low-Medium,Not Integrated,Currently no direct data access; relies on ret...
1,TXN000002,CUST00642,2024-01-01,Consumer Products,Merchandise - Apparel,40.87,3rd Party - Target,West Coast,3rd Party,Consumer Products,...,7,Consumer Products,3rd Party - Target,3rd Party,No,No,No,Low-Medium,Not Integrated,Currently no direct data access; relies on ret...
2,TXN000003,CUST00035,2024-01-01,Streaming,Monthly Subscription,10.72,1st Party - Streaming Platform,Southwest,1st Party,Streaming,...,1,Streaming,1st Party - Streaming Platform,1st Party,Yes,Yes,Yes (80% complete),High,Fully Integrated,Complete customer profiles with viewing history
3,TXN000004,CUST00163,2024-01-02,Theatrical,Mystery Manor,17.24,3rd Party - AMC Theatres,Southwest,3rd Party,Theatrical,...,4,Theatrical,3rd Party - AMC Theatres,3rd Party,Limited (cinema loyalty ID only),No,No,Medium,Limited - Aggregated Data Only,Currently receives only title-level sales data...
4,TXN000005,CUST00185,2024-01-02,Theatrical,Space Odyssey 2,27.81,3rd Party - Cinemark,East Coast,3rd Party,Theatrical,...,6,Theatrical,3rd Party - Cinemark,3rd Party,Limited (cinema loyalty ID only),No,No,Medium,Limited - Aggregated Data Only,Currently receives only title-level sales data...


In [14]:

df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
df['Revenue'] = pd.to_numeric(df['Revenue'], errors='coerce')



ref_date = pd.Timestamp('2025-01-01')

rfm = df.groupby('Customer ID').agg(
    Recency  =('Transaction Date', lambda x: (ref_date - x.max()).days),
    Frequency=('Transaction ID',   'count'),
    Monetary =('Revenue',          'sum')
).reset_index()

# Score 1–3 (3 = best for each dimension)
rfm['R_Score']   = pd.qcut(rfm['Recency'],                     3, labels=[3,2,1]).astype(int)
rfm['F_Score']   = pd.qcut(rfm['Frequency'].rank(method='first'), 3, labels=[1,2,3]).astype(int)
rfm['M_Score']   = pd.qcut(rfm['Monetary'].rank(method='first'),  3, labels=[1,2,3]).astype(int)
rfm['RFM_Total'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

def rfm_segment(score):
    if score >= 8:   return 'Champions'
    elif score >= 6: return 'Loyal'
    elif score >= 4: return 'At Risk'
    else:            return 'Low Value'

rfm['Segment'] = rfm['RFM_Total'].apply(rfm_segment)
rfm.head()

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Total,Segment
0,CUST00001,43,7,708.41,2,2,2,6,Loyal
1,CUST00002,47,8,1283.69,2,2,3,7,Loyal
2,CUST00003,178,3,124.72,1,1,1,3,Low Value
3,CUST00004,16,9,840.39,3,2,3,8,Champions
4,CUST00005,90,4,412.37,1,1,2,4,At Risk


1. Distribution of R, F, and M individually
2. RFM score heatmap
3. 3-digit RFM combination counts??? redo the metric?
4. Bubble or scatter plot: Frequency vs Monetary
5. Revenue contribution by segment
6. Cumulative revenue concentration within RFM